In [6]:
import numpy as np
from collections import deque
import time
import random
import math

In [7]:
class QuoridorState:
    def __init__(self):
        self.p1 = (8, 4)  # Row, Col (P1 starts bottom, goal 0)
        self.p2 = (0, 4)  # Row, Col (P2 starts top, goal 8)
        self.walls = set() # Set of (r, c, orient)
        self.turn = 0      # 0 = P1, 1 = P2
        self.p1_walls = 10
        self.p2_walls = 10
        self.winner = None

    def clone(self):
        new_state = QuoridorState()
        new_state.p1 = self.p1
        new_state.p2 = self.p2
        new_state.walls = set(self.walls) 
        new_state.turn = self.turn
        new_state.p1_walls = self.p1_walls
        new_state.p2_walls = self.p2_walls
        new_state.winner = self.winner
        return new_state

    def current_player(self):
        return self.turn

    def is_terminal(self):
        return self.winner is not None

    def returns(self):
        if self.winner == 0: return {0: 1.0, 1: -1.0}
        if self.winner == 1: return {0: -1.0, 1: 1.0}
        return {0: 0.0, 1: 0.0}

    def apply_action(self, action):
        if action < 4:
            # --- PAWN MOVE ---
            curr = self.p1 if self.turn == 0 else self.p2
            opp = self.p2 if self.turn == 0 else self.p1
            dr, dc = [(-1, 0), (0, 1), (1, 0), (0, -1)][action]
            
            nr, nc = curr[0] + dr, curr[1] + dc
            
            # Jump Logic
            if (nr, nc) == opp:
                nr, nc = nr + dr, nc + dc
            
            if self.turn == 0: self.p1 = (nr, nc)
            else: self.p2 = (nr, nc)
            
        else:
            # --- WALL PLACEMENT ---
            wall_idx = action - 4
            orient = wall_idx % 2
            idx = wall_idx // 2
            c = idx % 8
            r = idx // 8
            
            self.walls.add((r, c, orient))
            if self.turn == 0: self.p1_walls -= 1
            else: self.p2_walls -= 1

        # Check Win
        if self.p1[0] == 0: self.winner = 0
        elif self.p2[0] == 8: self.winner = 1
        
        self.turn = 1 - self.turn

    # --- ROBUST COLLISION LOGIC ---
    def is_blocked(self, start, end):
        r1, c1 = start
        r2, c2 = end
        
        # Vertical Move (Row changes)
        if c1 == c2: 
            r = min(r1, r2)
            # Blocked if wall at (r, c1, Horizontal) OR (r, c1-1, Horizontal)
            return (r, c1, 0) in self.walls or (r, c1 - 1, 0) in self.walls
            
        # Horizontal Move (Col changes)
        else: 
            c = min(c1, c2)
            # Blocked if wall at (r1, c, Vertical) OR (r1-1, c, Vertical)
            return (r1, c, 1) in self.walls or (r1 - 1, c, 1) in self.walls

    # --- ROBUST PATHFINDING ---
    def path_exists(self, start, goal_row):
        """Standard BFS to verify connectivity."""
        queue = deque([start])
        visited = {start}
        
        while queue:
            r, c = queue.popleft()
            if r == goal_row: return True
            
            # Check all 4 neighbors
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < 9 and 0 <= nc < 9:
                    if (nr, nc) not in visited:
                        # Crucial: Check if wall blocks this specific step
                        if not self.is_blocked((r, c), (nr, nc)):
                            visited.add((nr, nc))
                            queue.append((nr, nc))
        return False

    def _get_wall_neighbors(self, r, c, orient):
        candidates = set()
        for dr in range(-2, 3):
            for dc in range(-2, 3):
                nr, nc = r + dr, c + dc
                if 0 <= nr < 8 and 0 <= nc < 8:
                    candidates.add((nr, nc))
        return candidates

    def get_legal_actions(self, heuristic_pruning=True):
        actions = []
        curr = self.p1 if self.turn == 0 else self.p2
        opp = self.p2 if self.turn == 0 else self.p1
        
        # 1. Pawn Moves
        for i, (dr, dc) in enumerate([(-1, 0), (0, 1), (1, 0), (0, -1)]):
            nr, nc = curr[0] + dr, curr[1] + dc
            if 0 <= nr < 9 and 0 <= nc < 9:
                if not self.is_blocked(curr, (nr, nc)):
                    if (nr, nc) == opp: # Jump
                        nnr, nnc = nr + dr, nc + dc
                        if 0 <= nnr < 9 and 0 <= nnc < 9 and not self.is_blocked((nr, nc), (nnr, nnc)):
                            actions.append(i)
                    else:
                        actions.append(i)

        # 2. Wall Placements
        walls_left = self.p1_walls if self.turn == 0 else self.p2_walls
        if walls_left > 0:
            candidates = set()
            
            if heuristic_pruning:
                # Add walls near players
                for p_r, p_c in [curr, opp]:
                    for dr in [-1, 0, 1]:
                        for dc in [-1, 0, 1]:
                            if 0 <= p_r+dr < 8 and 0 <= p_c+dc < 8:
                                candidates.add((p_r+dr, p_c+dc))
                
                # Add walls near existing walls (to extend chains)
                for (w_r, w_c, w_o) in self.walls:
                    candidates.update(self._get_wall_neighbors(w_r, w_c, w_o))
            else:
                # Brute force all positions
                for r in range(8):
                    for c in range(8):
                        candidates.add((r, c))

            for r, c in candidates:
                for orient in [0, 1]:
                    # Overlap Check
                    if (r, c, 0) in self.walls or (r, c, 1) in self.walls: continue
                    if orient == 0 and ((r, c-1, 0) in self.walls or (r, c+1, 0) in self.walls): continue
                    if orient == 1 and ((r-1, c, 1) in self.walls or (r+1, c, 1) in self.walls): continue
                    
                    # --- CRITICAL PATH CHECK ---
                    self.walls.add((r, c, orient))
                    
                    # Must verify BOTH players still have a path
                    p1_safe = self.path_exists(self.p1, 0)
                    p2_safe = self.path_exists(self.p2, 8)
                    
                    self.walls.remove((r, c, orient))
                    
                    if p1_safe and p2_safe:
                        actions.append(4 + (r * 8 + c) * 2 + orient)
        
        return actions

    def get_pawn_moves_only(self):
        """Rollout Optimization"""
        actions = []
        curr = self.p1 if self.turn == 0 else self.p2
        opp = self.p2 if self.turn == 0 else self.p1
        for i, (dr, dc) in enumerate([(-1, 0), (0, 1), (1, 0), (0, -1)]):
            nr, nc = curr[0] + dr, curr[1] + dc
            if 0 <= nr < 9 and 0 <= nc < 9:
                if not self.is_blocked(curr, (nr, nc)):
                    if (nr, nc) == opp:
                        nnr, nnc = nr + dr, nc + dc
                        if 0 <= nnr < 9 and 0 <= nnc < 9 and not self.is_blocked((nr, nc), (nnr, nnc)):
                            actions.append(i)
                    else:
                        actions.append(i)
        return actions
    
    def get_shortest_path_len(self, start_pos, goal_row):
        """BFS to find the shortest distance to the goal row."""
        queue = deque([(start_pos, 0)])
        visited = {start_pos}
        
        while queue:
            (r, c), dist = queue.popleft()
            if r == goal_row:
                return dist
            
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < 9 and 0 <= nc < 9:
                    if (nr, nc) not in visited and not self.is_blocked((r, c), (nr, nc)):
                        visited.add((nr, nc))
                        queue.append(((nr, nc), dist + 1))
        return 999 # Should not happen if path exists

    def heuristic_value(self):
        """
        Returns the board value from the perspective of the current player.
        Positive = Winning, Negative = Losing.
        """
        # 1. Calculate Distances
        p1_dist = self.get_shortest_path_len(self.p1, 0)
        p2_dist = self.get_shortest_path_len(self.p2, 8)
        
        # 2. Score = (Opponent Dist - My Dist)
        # If I am P1 (Goal 0), I want p1_dist small and p2_dist large.
        if self.turn == 0:
            score = p2_dist - p1_dist
        else:
            score = p1_dist - p2_dist
            
        # 3. Normalize to range [-1, 1] for MCTS
        # Max path diff is roughly 20-30 moves. 
        return max(-1.0, min(1.0, score / 10.0))
    
    def __str__(self):
        return str((self.p1, self.p2, frozenset(self.walls), self.turn))

In [8]:
class MCTSNode:
    def __init__(self, state, parent=None, parent_action=None):
        self.state = state
        self.children = {}
        self.visits = 0
        self.value = 0.0
        
        # --- FIX ---
        # Get actions
        self.untried_actions = state.get_legal_actions(heuristic_pruning=True)[::-1]

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, c_param=1.414):
        best_score = float('-inf')
        best_child = None
        for action, child in self.children.items():
            if child.visits == 0: return child # Avoid div by zero
            
            exploit = child.value / child.visits
            explore = math.sqrt(math.log(self.visits) / child.visits)
            score = exploit + c_param * explore
            
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

In [9]:
class MCTS_Enhanced:
    def __init__(self, iterations=1000):
        self.iterations = iterations
        self.tt = {} # Transposition Table

    def get_node(self, state):
        # We don't store parent in the node anymore because it's ambiguous in a DAG
        state_key = str(state)
        if state_key not in self.tt:
            self.tt[state_key] = MCTSNode(state.clone())
        return self.tt[state_key]

    def search(self, root_state):
        root = self.get_node(root_state)

        for i in range(self.iterations):
            # 1. Explicitly track the path for this specific iteration
            # This replaces the unreliable 'node.parent' pointer
            search_path = [root]
            node = root
            
            # --- SELECTION ---
            while node.is_fully_expanded() and not node.state.is_terminal():
                # Cycle Detection (using path)
                if node in search_path[:-1]:
                    break
                
                next_node = node.best_child()
                
                # Dead End Check (Fixes your previous NoneType error)
                if next_node is None: 
                    break 
                
                node = next_node
                search_path.append(node)
            
            # --- EXPANSION ---
            if not node.state.is_terminal() and not node.is_fully_expanded():
                action = node.untried_actions.pop()
                next_state = node.state.clone()
                next_state.apply_action(action)
                
                child_node = self.get_node(next_state)
                node.children[action] = child_node
                search_path.append(child_node)
                
                # Update current node to child for simulation
                node = child_node
            
            # --- SIMULATION (BIASED ROLLOUT) ---
            rollout_state = node.state.clone()
            depth = 0
            
            while not rollout_state.is_terminal() and depth < 40:
                # Decide: Pawn Move or Wall?
                # 20% chance to consider walls (Simulate tactical play)
                # 80% chance to just race pawns (Speed optimization)
                if random.random() < 0.2 and (rollout_state.p1_walls > 0 or rollout_state.p2_walls > 0):
                    # Use your optimized heuristic to find RELEVANT walls only
                    actions = rollout_state.get_legal_actions(heuristic_pruning=True)
                else:
                    # Fast path: Pawn moves only
                    actions = rollout_state.get_pawn_moves_only()
                
                if not actions: break
                
                # Bias: Prefer moving forward to random wall placement?
                # For simplicity, random choice is fine for rollout
                action = random.choice(actions) 
                
                rollout_state.apply_action(action)
                depth += 1

            # --- BACKPROPAGATION (ALONG PATH) ---
            # We walk back up the search_path list we built
            returns = rollout_state.returns()
            
            for idx in range(len(search_path) - 1, -1, -1):
                curr_node = search_path[idx]
                curr_node.visits += 1
                
                # Update value based on the move that led here
                # The node at search_path[idx] was chosen by search_path[idx-1]
                if idx > 0:
                    parent_node = search_path[idx-1]
                    # Who made the move to get to curr_node?
                    player_who_moved = parent_node.state.current_player()
                    # Add reward from that player's perspective
                    curr_node.value += returns[player_who_moved]
        
        # Select best move
        if not root.children: return None 
        return max(root.children.items(), key=lambda item: item[1].visits)[0]

In [10]:
def train_self_play(ai_agent, num_games=10):
    print(f"Starting Self-Play Training ({num_games} games)...")
    p1_wins = 0
    p2_wins = 0
    
    for i in range(num_games):
        state = QuoridorState()
        moves = 0
        
        while not state.is_terminal() and moves < 200:
            # Both players use the SAME AI agent
            # The agent automatically handles perspective based on state.turn
            
            # 1. Search
            best_action = ai_agent.search(state)
            
            # 2. Selection (Temperature) for Training Diversity
            # If we always pick max(), every game will be identical.
            # We add noise for the first 6 moves to explore different openings.
            if moves < 6:
                # Get visit counts from the root's children
                root_node = ai_agent.tt[str(state)]
                children = list(root_node.children.items())
                
                if children:
                    actions = [a for a, n in children]
                    visits = [n.visits for a, n in children]
                    total_visits = sum(visits)
                    probs = [v / total_visits for v in visits]
                    
                    # Sample an action based on visit probability
                    action = np.random.choice(actions, p=probs)
                else:
                    action = best_action
            else:
                # Play strictly best move for the rest of the game
                action = best_action
            
            state.apply_action(action)
            moves += 1
            
            # Optional: Print progress dots
            if moves % 10 == 0: print(".", end="", flush=True)
            
        # Game Over
        print(f" Game {i+1} finished in {moves} moves. Winner: P{state.winner}")
        
        if state.winner == 0: p1_wins += 1
        else: p2_wins += 1

    print(f"\nTraining Complete.")
    print(f"Final Stats in TT: {len(ai_agent.tt)} unique states memorized.")
    print(f"P1 Wins: {p1_wins}, P2 Wins: {p2_wins}")

# --- EXECUTE TRAINING ---
# Initialize a fresh AI
ai_brain = MCTS_Enhanced(iterations=1000)

# Train it against itself
train_self_play(ai_brain, num_games=30)

Starting Self-Play Training (30 games)...


KeyboardInterrupt: 

In [ ]:
def action_to_string(action):
    """Translates integer action to human-readable string."""
    if action == 0: return "Move Up"
    if action == 1: return "Move Right"
    if action == 2: return "Move Down"
    if action == 3: return "Move Left"
    
    # Wall decoding
    wall_idx = action - 4
    orient = wall_idx % 2
    idx = wall_idx // 2
    c = idx % 8
    r = idx // 8
    
    o_str = "Horizontal" if orient == 0 else "Vertical"
    # Note: Returns (row, col) of the top-left intersection
    return f"Wall at ({r}, {c}) {o_str}"

def string_to_action(input_str):
    """
    Parses human input into integer action.
    Formats:
      - 'u', 'd', 'l', 'r' (Moves)
      - 'w 3 4 h' (Wall at row 3, col 4, horizontal)
      - 'w 2 5 v' (Wall at row 2, col 5, vertical)
    """
    parts = input_str.strip().lower().split()
    if not parts: return None
    
    cmd = parts[0]
    
    # 1. Moves
    if cmd in ['u', 'up']: return 0
    if cmd in ['r', 'right']: return 1
    if cmd in ['d', 'down']: return 2
    if cmd in ['l', 'left']: return 3
    
    # 2. Walls
    # Expected format: "w [row] [col] [h/v]"
    if cmd in ['w', 'wall'] and len(parts) >= 4:
        try:
            r = int(parts[1])
            c = int(parts[2])
            orient_char = parts[3]
            orient = 0 if orient_char.startswith('h') else 1
            
            # Re-encode: 4 + (r * 8 + c) * 2 + orient
            if 0 <= r < 8 and 0 <= c < 8:
                return 4 + (r * 8 + c) * 2 + orient
        except ValueError:
            return None
            
    return None

def print_board(state):
    """Visualizes the board state with ASCII."""
    # Create 17x17 grid (cells + spaces for walls)
    # 9 cells -> indices 0, 2, 4... 16
    board_viz = [[' ' for _ in range(17)] for _ in range(17)]
    
    # Mark Cells
    for r in range(9):
        for c in range(9):
            board_viz[r*2][c*2] = '.'
            
    # Mark Players
    p1_r, p1_c = state.p1
    p2_r, p2_c = state.p2
    board_viz[p1_r*2][p1_c*2] = '1'
    board_viz[p2_r*2][p2_c*2] = '2'
    
    # Mark Walls
    for (r, c, orient) in state.walls:
        # r,c is the top-left intersection. 
        # In grid terms: intersection is at row 2r+1, col 2c+1
        ir, ic = r*2 + 1, c*2 + 1
        
        board_viz[ir][ic] = '+' # The intersection center
        
        if orient == 0: # Horizontal
            board_viz[ir][ic-1] = '-'
            board_viz[ir][ic+1] = '-'
        else: # Vertical
            board_viz[ir-1][ic] = '|'
            board_viz[ir+1][ic] = '|'

    print("\n  " + " ".join([str(i) for i in range(9)]))
    for i in range(17):
        row_label = str(i//2) if i % 2 == 0 else " "
        print(f"{row_label} " + "".join(board_viz[i]))

In [ ]:
# Ensure your AI is initialized
# ai = MCTS_Enhanced(iterations=1000) 

def human_vs_ai(ai_agent):
    game_state = QuoridorState()
    print("--- Human (P2 - Top) vs AI (P1 - Bottom) ---")
    print("Controls:")
    print("  Moves: 'u' (up), 'd' (down), 'l' (left), 'r' (right)")
    print("  Walls: 'w r c h' (row col hor) | e.g., 'w 4 4 h'")
    
    while not game_state.is_terminal():
        print_board(game_state)
        
        if game_state.current_player() == 0:
            # AI TURN
            print("\nAI is thinking...")
            action = ai_agent.search(game_state)
            print(f"AI plays: {action_to_string(action)}")
            game_state.apply_action(action)
            
        else:
            # HUMAN TURN
            print("\n--- Your Turn (Player 2) ---")
            
            # Get legal actions to validate input
            legal_ints = game_state.get_legal_actions(heuristic_pruning=True)
            
            while True:
                user_input = input("Enter move: ")
                action = string_to_action(user_input)
                
                if action is None:
                    print("Invalid format. Use 'u', 'd', 'l', 'r' or 'w r c h'.")
                elif action not in legal_ints:
                    print(f"Illegal move! That action is not allowed. ({action_to_string(action)})")
                else:
                    game_state.apply_action(action)
                    break

    print_board(game_state)
    print("Game Over!")
    print("Winner:", "AI" if game_state.winner == 0 else "Human")

# Run the game
human_vs_ai(ai_brain)

--- Human (P2 - Top) vs AI (P1 - Bottom) ---
Controls:
  Moves: 'u' (up), 'd' (down), 'l' (left), 'r' (right)
  Walls: 'w r c h' (row col hor) | e.g., 'w 4 4 h'

  0 1 2 3 4 5 6 7 8
0 . . . . 2 . . . .
                   
1 . . . . . . . . .
                   
2 . . . . . . . . .
                   
3 . . . . . . . . .
                   
4 . . . . . . . . .
                   
5 . . . . . . . . .
                   
6 . . . . . . . . .
                   
7 . . . . . . . . .
                   
8 . . . . 1 . . . .

AI is thinking...
AI plays: Move Up

  0 1 2 3 4 5 6 7 8
0 . . . . 2 . . . .
                   
1 . . . . . . . . .
                   
2 . . . . . . . . .
                   
3 . . . . . . . . .
                   
4 . . . . . . . . .
                   
5 . . . . . . . . .
                   
6 . . . . . . . . .
                   
7 . . . . 1 . . . .
                   
8 . . . . . . . . .

--- Your Turn (Player 2) ---

  0 1 2 3 4 5 6 7 8
0 . . . . . . . . .
         